# MS MARCO RARS-v7 Frozen-Index Query Adapter Pilot

## Goal

Run the frozen single-seed V7 pilot at implementation commit `b9c930508d1e6f95a191983dc2a8e2027bc33c36`. The experiment trains only a 48 KiB query-side rank-16 adapter. The million-document IVF-PQ index, document embeddings, list membership, codebooks, and PQ codes remain immutable.

A GO is development evidence only. It authorizes a separately preregistered development audit, not RARS combination, future-holdout access, an official MS MARCO claim, or a paper claim.

## Experiment contract

The original query selects the same 16 IVF lists as V6. The adapted query scores immutable PQ reconstructions inside those lists. Training uses a label-blind, query-disjoint 1,845/462 split of the already outcome-informed `oracle_design` role. Top-100 promotion and Top-10 protection pairs use explicit-positive versus unjudged-challenger semantics.

Before training, the durable V6 packet must pass byte/hash verification and metric/gate recomputation. Epoch zero must reproduce the V6 Base-PQ and same-route FP32 Recall arrays exactly.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v7-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
EXPERIMENT_ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '05c2ae43b7d11783460822d10c590240dab1a399'
V6_IMPLEMENTATION_COMMIT = '26a7717b964eed979b3bf7a3149d0d24e9bce3f1'
V7_IMPLEMENTATION_COMMIT = 'b9c930508d1e6f95a191983dc2a8e2027bc33c36'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3')
V7_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v7')
PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
V3_WORK = Path('/content') / f'rars-v3-{V3_IMPLEMENTATION_COMMIT[:12]}'
for work in (PARENT_WORK, V3_WORK):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = V3_WORK / 'bundles'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
V6_PACKET = DRIVE / 'rars-v6-1m-headroom' / V6_IMPLEMENTATION_COMMIT[:12]
OUTPUT = DRIVE / 'rars-v7-query-adapter-pilot' / V7_IMPLEMENTATION_COMMIT[:12] / 'seed42'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

In [ ]:
clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)
clone_exact(V7_REPO, V7_IMPLEMENTATION_COMMIT)

V3_PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
V7_PROTOCOL_PATH = V7_REPO / 'protocols/rars_v7_query_adapter_pilot_v1.json'
protocol = json.loads(V7_PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_V7_TRAINING_RUN'
assert protocol['precondition']['v6_source_commit'] == V6_IMPLEMENTATION_COMMIT
assert protocol['adapter']['side'] == 'query_only'
assert protocol['frozen_index_contract']['pq_codes_immutable'] is True
assert protocol['positioning']['rars_combination_locked'] is True

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
    'tests/test_rars_v3_oracle_protocol_contract.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v7_query_adapter_core.py',
    'tests/test_rars_v7_query_adapter_protocol_contract.py',
    'tests/test_train_rars_v7_query_adapter_contract.py',
    'tests/test_verify_rars_v6_1m_headroom_packet.py',
    'tests/test_rars_v6_headroom_core.py',
    'tests/test_evaluate_rars_v6_1m_headroom.py',
], cwd=V7_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact V7 implementation commit:', V7_IMPLEMENTATION_COMMIT)
print('Frozen V7 protocol SHA-256:', sha256_file(V7_PROTOCOL_PATH))

In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
    V6_PACKET / 'headroom_result.json',
    V6_PACKET / 'headroom_complete.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 8_000_000_000, 'Need 8 GB local disk'
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), (
    'The durable V7 output is non-empty. Do not overwrite it; return it for audit.'
)
v6_verification = subprocess.check_output([
    EXPERIMENT_PYTHON, str(V7_REPO / 'scripts/verify_rars_v6_1m_headroom_packet.py'),
    '--packet-root', str(V6_PACKET),
], text=True, cwd=V7_REPO, env=EXPERIMENT_ENV)
v6_summary = json.loads(v6_verification)
assert v6_summary['status'] == 'RARS_V6_1M_HEADROOM_PACKET_VERIFIED'
assert v6_summary['formal_decision'] == 'GO_TO_V6_LOSS_IMPLEMENTATION'
assert v6_summary['source_commit'] == V6_IMPLEMENTATION_COMMIT
print(json.dumps(v6_summary, indent=2, allow_nan=False))

## Rebuild the registered design identity

The next cells rematerialize the exact v2.2 parent and the pinned qrels-free v3 split. The v3 builder writes qrels-free design/audit candidate artifacts together, but no role labels are materialized. Only `oracle_design` is passed to V7; the trainer never opens `oracle_audit`, and `future_method_holdout` remains identity-only.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
print('Exact v2.2 parent rematerialized.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(V3_PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
candidate_summary = json.loads((V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text())
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['qrels_opened_or_parsed'] is False
ROLE_LABEL_FILES = {
    'candidate_relevance.uint8.npy', 'relevant_counts.int32.npy',
    'v3_role_labels_started.json', 'v3_role_labels_manifest.json',
}
for role in ('oracle_design', 'oracle_audit'):
    assert not ROLE_LABEL_FILES.intersection(path.name for path in (V3_BUNDLES / role).iterdir())
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}
print('Design identity ready; audit unlabeled; future identity-only.')

## Run the frozen seed-42 pilot

The trainer verifies V6 again, fixes the original IVF routes, recreates the V6 flip population, performs a label-blind design train/selection split, and checks epoch-zero parity before the first gradient step. Do not interrupt this cell or edit either durable packet.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    EXPERIMENT_PYTHON, str(V7_REPO / 'scripts/train_rars_v7_query_adapter.py'),
    '--design-role-dir', str(V3_BUNDLES / 'oracle_design'),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--index', str(INDEX),
    '--v6-packet-root', str(V6_PACKET),
    '--output-dir', str(OUTPUT),
    '--protocol', str(V7_PROTOCOL_PATH),
    '--source-commit', V7_IMPLEMENTATION_COMMIT,
], check=True, cwd=V7_REPO, env=EXPERIMENT_ENV)
print('RARS-v7 seed-42 pilot completed.')

In [ ]:
complete_path = OUTPUT / 'training_complete.json'
result_path = OUTPUT / 'pilot_result.json'
complete = json.loads(complete_path.read_text())
result = json.loads(result_path.read_text())
assert complete['status'] == 'V7_QUERY_ADAPTER_PILOT_COMPLETE'
assert result['status'] == 'V7_QUERY_ADAPTER_PILOT_COMPLETE'
assert complete['source_commit'] == V7_IMPLEMENTATION_COMMIT
assert result['source_commit'] == V7_IMPLEMENTATION_COMMIT
assert complete['formal_decision'] == result['formal_decision']
assert complete['formal_decision'] in {'GO_TO_V7_DEVELOPMENT_AUDIT', 'STOP_V7_QUERY_ADAPTER_PILOT'}
assert complete['index_before'] == complete['index_after']
assert complete['document_reencoding_performed'] is False
assert complete['rars_used'] is False
assert complete['oracle_audit_opened'] is False
assert complete['future_method_holdout_opened'] is False
assert result['selection']['query_count'] == 462
for field, filename in (
    ('started', 'training_started.json'), ('split', 'split_manifest.json'),
    ('history', 'training_history.json'), ('result', 'pilot_result.json'),
):
    verify_record(OUTPUT / filename, complete[field])
for filename, record in complete['outputs'].items():
    verify_record(OUTPUT / filename, record)
missing_outputs = [name for name in protocol['required_outputs'] if not (OUTPUT / name).is_file()]
assert not missing_outputs, missing_outputs
report = {
    'formal_decision': result['formal_decision'],
    'selected_epoch': result['selected_epoch'],
    'epochs_executed': result['epochs_executed'],
    'selection': result['selection'],
    'failed_gates': result['decision']['failed_gates'],
    'training_pair_support': result['training_pair_support'],
    'selection_pair_support_diagnostic_only': result['selection_pair_support_diagnostic_only'],
    'telemetry': result['telemetry'],
    'result_sha256': sha256_file(result_path),
}
print(json.dumps(report, indent=2, allow_nan=False))

## Checks and next steps

Return the final printed report plus `pilot_result.json` and `training_complete.json` for audit. If the decision is `STOP_V7_QUERY_ADAPTER_PILOT`, do not tune this frozen run after seeing selection outcomes. If it is `GO_TO_V7_DEVELOPMENT_AUDIT`, freeze a new audit protocol around this exact checkpoint; do not open the future holdout or combine RARS yet.